# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [5]:
import pandas as pd

# Load the dataset
url = 'https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv'
df = pd.read_csv(url)

# Display the first few rows of the DataFrame
df.head()

,Unnamed: 0,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,...,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size,Vehicle Type
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2/18/11,Employed,M,...,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,1,KX64629,California,2228.525238,No,Basic,College,1/18/11,Unemployed,F,...,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2/10/11,Employed,M,...,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,1/17/11,Medical Leave,F,...,NaN,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,NaN


In [43]:
df.columns = df.columns.str.lower().str.strip()

df_filtered = df[(df['total_claim_amount'] < 1000) & (df['response'] == 'Yes')]
df.loc[(df['total_claim_amount'] < 1000) & (df['response'] == 'Yes'), 'low_total_claim_amount'] = 'low_total_claim_amount'

print(f"Clientes encontrados: {len(df_filtered)}")
df_filtered.head()

Clientes encontrados: 1399


,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type,low_total_claim_amount,response_rate
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A,low_total_claim_amount,1
8,8,FM55990,California,5989.773931,Yes,Premium,College,1/19/11,Employed,M,...,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,NaN,low_total_claim_amount,1
15,15,CW49887,California,4626.801093,Yes,Basic,Master,1/16/11,Employed,F,...,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,NaN,low_total_claim_amount,1
19,19,NJ54277,California,3746.751625,Yes,Extended,College,2/26/11,Employed,F,...,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A,low_total_claim_amount,1
27,27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2/28/11,Employed,F,...,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,NaN,low_total_claim_amount,1


In [50]:

# 2. Normalize the 'response' column values to ensure 'Yes' is caught regardless of case
df['response'] = df['response'].str.strip().str.capitalize()


# 3. Filter for active respondents
df_respondents = df[df['response'] == 'Yes']

# 4. Comprehensive Segment Analysis
# We now use 'monthly_premium' as explicitly requested by the exercise
segment_analysis = df_respondents.groupby(['policy_type', 'gender'])[[
    'monthly_premium', 
    'customer_lifetime_value', 
    'total_claim_amount'
]].mean().round(2)

# 5. Calculate the 'Profit Proxy' (Monthly Premium - Monthly Claim average)
# We divide total_claim_amount by 12 to normalize it to a monthly cost
segment_analysis['profit_proxy'] = (
    segment_analysis['monthly_premium'] - (segment_analysis['total_claim_amount'] / 12)
).round(2)

# 6. Sort by profit_proxy to highlight the most attractive segments
segment_analysis = segment_analysis.sort_values(by='profit_proxy', ascending=False)

print("--- Segment Profitability and Risk Analysis ---")
print(segment_analysis)

--- Segment Profitability and Risk Analysis ---
                       monthly_premium  customer_lifetime_value  \
policy_type    gender                                             
Personal Auto  F                 99.00                  8339.79   
Corporate Auto F                 94.30                  7712.63   
               M                 92.19                  7944.47   
Special Auto   F                 92.31                  7691.58   
Personal Auto  M                 91.09                  7448.38   
Special Auto   M                 86.34                  8247.09   

                       total_claim_amount  profit_proxy  
policy_type    gender                                    
Personal Auto  F                   452.97         61.25  
Corporate Auto F                   433.74         58.15  
               M                   408.58         58.14  
Special Auto   F                   453.28         54.54  
Personal Auto  M                   457.01         53.01  
Special A

Women with a Personal Auto Policy who responded "Yes" pay the highest premiums ($99) and maintain a very solid Customer Lifetime Value (CLV). Combined with their presence in the low-claims segment, they represent the ideal customer profile.

The Special Auto (M) segment represents the lowest risk; despite having lower premiums, they exhibit a high CLV. This suggests they are stable, long-term policyholders who rarely cause issues. Generally, customers who respond "Yes" tend to have a slightly higher Customer Lifetime Value than those who do not.














































In [ ]:

# 1. Ensure column names are clean (lowercase and no spaces)
df.columns = df.columns.str.lower().str.strip()


state_counts = df.groupby('state')['customer'].count()

# 3. Filter for states that have more than 500 customers
# This identifies the "high-volume" regions
high_volume_states = state_counts[state_counts > 500]

# 4. Display the result
print("States with more than 500 customers:")
print(high_volume_states)

States with more than 500 customers:
state
Arizona       1937
California    3552
Nevada         993
Oregon        2909
Washington     888
Name: customer, dtype: int64


In [35]:
# 1. Agrupamos por nivel educativo y género
df_yes = df[df['response'] == 'Yes']
# 2. Calculamos la mediana del valor de vida del cliente (CLV)
analysis_median = df_yes.groupby(['education', 'gender'])[['customer_lifetime_value']].median()

# Mostramos el resultado por pantalla
print(analysis_median)

                             customer_lifetime_value
education            gender                         
Bachelor             F                   5246.278375
                     M                   5548.031892
College              F                   4834.710493
                     M                   5989.773931
Doctor               F                   4673.457903
                     M                   5073.282126
High School or Below F                   6503.397049
                     M                   6366.225775
Master               F                   5096.673223
                     M                   8509.850887


In [36]:
analysis_maximum = df_yes.groupby(['education', 'gender'])[['customer_lifetime_value']].max()

# Mostramos el resultado por pantalla
print(analysis_maximum)

                             customer_lifetime_value
education            gender                         
Bachelor             F                  33473.349460
                     M                  24127.504020
College              F                  25807.063000
                     M                  22332.439460
Doctor               F                   6265.343299
                     M                  12731.951610
High School or Below F                  41787.903430
                     M                  12298.686300
Master               F                  19160.989940
                     M                  14435.673650


In [ ]:
analysis_minimum = df_yes.groupby(['education', 'gender'])[['customer_lifetime_value']].min()

# Mostramos el resultado por pantalla
print(analysis_minimum)

                             customer lifetime value
education            gender                         
Bachelor             F                   2248.449633
                     M                   2300.691547
College              F                   2004.350666
                     M                   2393.915369
Doctor               F                   2395.570000
                     M                   2491.317024
High School or Below F                   2397.036098
                     M                   2227.072755
Master               F                   2574.020376
                     M                   2551.226692


High-Value Outliers: The highest individual CLV is found among females with High School or Below education (over $41,787), followed by females with a Bachelor's degree ($33,473). This suggests that some of the most loyal/profitable customers do not necessarily hold the highest academic degrees.

The "Doctor" Paradox: Interestingly, customers with a Doctorate degree show the lowest maximum CLV ($12,731 for males and only $6,265 for females), and their median value is among the lowest. This segment appears to be the least profitable for this specific marketing response.

Consistency: The minimum CLV is remarkably stable across all segments (around $2,000 - $2,500), indicating a consistent "floor" for customer value regardless of demographic.

Top Performers by Median: Male Masters graduates show the highest median CLV ($8,509), suggesting that while they might not have the extreme outliers of other groups, they are consistently high-value custome

In [27]:
# 1. Ensure columns are clean
df.columns = df.columns.str.lower().str.strip()

# 2. Group by State and Month, counting a unique identifier (like 'customer' or 'policy_id')
# This ensures we are counting actual policy records
policy_counts = df.groupby(['state', 'effective to date'])['customer'].count()

# 3. Pivot the data (unstack) and clean up
# we fillna(0) because a missing value means zero policies were sold
policy_table = policy_counts.unstack().fillna(0).astype(int)

# 4. Optional: Sort by total sales to see the top states at a glance
policy_table['total'] = policy_table.sum(axis=1)
policy_table = policy_table.sort_values(by='total', ascending=False)

print("Policies Sold per State and Month:")
print(policy_table)

Policies Sold per State and Month:
effective to date  1/1/11  1/10/11  1/11/11  1/12/11  1/13/11  1/14/11  \
state                                                                    
California             56       72       81       49       55       57   
Oregon                 42       70       41       36       39       54   
Arizona                35       34       35       32       35       36   
Nevada                 13       25       19       12       20       10   
Washington             18       21        9       12       13       11   

effective to date  1/15/11  1/16/11  1/17/11  1/18/11  ...  2/27/11  2/28/11  \
state                                                  ...                     
California              54       66       60       59  ...       64       49   
Oregon                  51       44       77       45  ...       55       64   
Arizona                 29       33       35       37  ...       40       29   
Nevada                  18       10       14  

In [29]:
# 1. Convert response to numeric (1 for Yes, 0 for No)
# This allows us to calculate the "Mean", which is the Response Rate
df['response_rate'] = df['response'].apply(lambda x: 1 if x == 'Yes' else 0)

# 2. Analyze by Marketing Channel (Sales Channel)
# We calculate the mean of our new 1/0 column
channel_analysis = df.groupby('sales channel')['response_rate'].mean().reset_index()

# 3. Convert to percentage for better readability
channel_analysis['response_rate'] *= 100

# 4. Sort to see the most effective channel at the top
channel_analysis = channel_analysis.sort_values(by='response_rate', ascending=False)

print("Response Rate (%) by Sales Channel:")
print(channel_analysis.round(2))

Response Rate (%) by Sales Channel:
  sales channel  response_rate
0         Agent          18.01
3           Web          10.89
1        Branch          10.79
2   Call Center          10.32
